# 03_collate_all_species.ipynb

Collates all cleaned GBIF species data into a single master GeoDataFrame,  
then splits into functional groups as defined in the connectivity framework.

**Excluded species** (no GBIF records or no data after cleaning):
- `Iberochondrostoma almacai` — no GBIF records
- `Sylvia melanocephala` — empty after coordinate uncertainty filter
- `Squalius torgalensis` — empty after coordinate uncertainty filter
- `Salmo salar` — empty after coordinate uncertainty filter

**Functional groups** (medium = connectivity element, AIR / LAND / WATER):
1. Terrestrial Carnivores (LAND)
2. Large Herbivores (LAND)
3. Birds (AIR)
4. Aquatic & Freshwater Species (WATER)
5. Pollinators (AIR)
6. Lepidoptera (AIR)
7. Vegetation (LAND matrix)
8. Decomposers (LAND)
9. Invasive Species (AIR)

Translocation species are flagged with `translocation = True`.

## Imports

In [6]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import numpy as np

## Project Paths

In [7]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_SPECIES = PROJECT_ROOT / 'data' / 'processed' / 'species'
PROCESSED_GROUPS  = PROJECT_ROOT / 'data' / 'processed' / 'groups'

PROCESSED_GROUPS.mkdir(parents=True, exist_ok=True)

print('Processed species dir :', PROCESSED_SPECIES)
print('Processed groups dir  :', PROCESSED_GROUPS)

Processed species dir : /home/linda/Documents/myData/data-management/data/processed/species
Processed groups dir  : /home/linda/Documents/myData/data-management/data/processed/groups


## Species Registry

Maps each species to:
- `folder_name` — subdirectory name under `data/processed/species/`
- `functional_group` — ecological functional group
- `medium` — connectivity medium (LAND / AIR / WATER)
- `translocation` — whether the species is part of the translocation framework

Species excluded (no records / empty after cleaning) are commented out.

In [8]:
SPECIES_REGISTRY = [

    # ── 1. Terrestrial Carnivores (LAND) ────────────────────────────────────
    dict(species='Canis lupus',           folder='Canis_lupus',              functional_group='Terrestrial Carnivores', medium='LAND', translocation=False),
    dict(species='Lynx pardinus',         folder='Lynx_pardinus',            functional_group='Terrestrial Carnivores', medium='LAND', translocation=True),
    dict(species='Felis silvestris',      folder='Felis_silvestris',         functional_group='Terrestrial Carnivores', medium='LAND', translocation=False),
    dict(species='Vulpes vulpes',         folder='Vulpes_vulpes',            functional_group='Terrestrial Carnivores', medium='LAND', translocation=False),
    dict(species='Genetta genetta',       folder='Genetta_genetta',          functional_group='Terrestrial Carnivores', medium='LAND', translocation=False),
    dict(species='Herpestes ichneumon',   folder='Herpestes_ichneumon',      functional_group='Terrestrial Carnivores', medium='LAND', translocation=False),
    dict(species='Martes foina',          folder='Martes_foina',             functional_group='Terrestrial Carnivores', medium='LAND', translocation=False),
    dict(species='Mustela putorius',      folder='Mustela_putorius',         functional_group='Terrestrial Carnivores', medium='LAND', translocation=False),

    # ── 2. Large Herbivores (LAND) ───────────────────────────────────────────
    dict(species='Cervus elaphus',        folder='Cervus_elaphus',           functional_group='Large Herbivores',       medium='LAND', translocation=True),
    dict(species='Capreolus capreolus',   folder='Capreolus_capreolus',      functional_group='Large Herbivores',       medium='LAND', translocation=True),
    dict(species='Capra pyrenaica',       folder='Capra_pyrenaica',          functional_group='Large Herbivores',       medium='LAND', translocation=False),
    dict(species='Sus scrofa',            folder='Sus_scrofa',               functional_group='Large Herbivores',       medium='LAND', translocation=False),
    dict(species='Lepus europaeus',       folder='Lepus_europaeus',          functional_group='Large Herbivores',       medium='LAND', translocation=False),
    dict(species='Oryctolagus cuniculus', folder='Oryctolagus_cuniculus',    functional_group='Large Herbivores',       medium='LAND', translocation=True),
    dict(species='Lepus granatensis',     folder='Lepus_granatensis',        functional_group='Large Herbivores',       medium='LAND', translocation=True),

    # ── 3. Birds (AIR) ───────────────────────────────────────────────────────
    dict(species='Aquila chrysaetos',     folder='Aquila_chrysaetos',        functional_group='Birds',                  medium='AIR',  translocation=False),
    dict(species='Bubo bubo',             folder='Bubo_bubo',                functional_group='Birds',                  medium='AIR',  translocation=False),
    dict(species='Gyps fulvus',           folder='Gyps_fulvus',              functional_group='Birds',                  medium='AIR',  translocation=False),
    dict(species='Neophron percnopterus', folder='N_percnopterus',           functional_group='Birds',                  medium='AIR',  translocation=False),
    dict(species='Pandion haliaetus',     folder='Pandion_haliaetus',        functional_group='Birds',                  medium='AIR',  translocation=True),
    dict(species='Porphyrio porphyrio',   folder='Porphyrio_porphyrio',      functional_group='Birds',                  medium='AIR',  translocation=True),
    dict(species='Alectoris rufa',        folder='Alectoris_rufa',           functional_group='Birds',                  medium='AIR',  translocation=False),
    dict(species='Turdus merula',         folder='Turdus_merula',            functional_group='Birds',                  medium='AIR',  translocation=False),
    dict(species='Turdus philomelos',     folder='Turdus_philomelos',        functional_group='Birds',                  medium='AIR',  translocation=False),
    # Sylvia melanocephala → EXCLUDED: empty after coordinate uncertainty filter

    # ── 4. Aquatic & Freshwater Species (WATER) ──────────────────────────────
    dict(species='Lutra lutra',                   folder='Lutra_lutra',                  functional_group='Aquatic & Freshwater', medium='WATER', translocation=False),
    dict(species='Pseudochondrostoma polylepis',   folder='P_polylepis',                  functional_group='Aquatic & Freshwater', medium='WATER', translocation=False),
    dict(species='Luciobarbus bocagei',            folder='L_bocagei',                    functional_group='Aquatic & Freshwater', medium='WATER', translocation=False),
    dict(species='Emys orbicularis',               folder='Emys_orbicularis',             functional_group='Aquatic & Freshwater', medium='WATER', translocation=False),
    dict(species='Salmo trutta',                   folder='Salmo_trutta',                 functional_group='Aquatic & Freshwater', medium='WATER', translocation=False),
    dict(species='Hippocampus hippocampus',        folder='H_hippocampus',                functional_group='Aquatic & Freshwater', medium='WATER', translocation=False),
    dict(species='Anaecypris hispanica',           folder='Anaecypris_hispanica',         functional_group='Aquatic & Freshwater', medium='WATER', translocation=False),
    dict(species='Achondrostoma occidentale',      folder='Achondrostoma_occidentale',    functional_group='Aquatic & Freshwater', medium='WATER', translocation=True),
    dict(species='Iberochondrostoma lusitanicum',  folder='I_lusitanicum',                functional_group='Aquatic & Freshwater', medium='WATER', translocation=True),
    # Iberochondrostoma almacai → EXCLUDED: no GBIF records
    dict(species='Squalius aradensis',             folder='Squalius_aradensis',           functional_group='Aquatic & Freshwater', medium='WATER', translocation=True),
    # Squalius torgalensis → EXCLUDED: empty after coordinate uncertainty filter
    dict(species='Squalius alburnoides',           folder='Squalius_alburnoides',         functional_group='Aquatic & Freshwater', medium='WATER', translocation=True),
    dict(species='Squalius pyrenaicus',            folder='Squalius_pyrenaicus',          functional_group='Aquatic & Freshwater', medium='WATER', translocation=True),
    dict(species='Margaritifera margaritifera',    folder='M_margaritifera',              functional_group='Aquatic & Freshwater', medium='WATER', translocation=True),
    # Salmo salar → EXCLUDED: empty after coordinate uncertainty filter

    # ── 5. Pollinators (AIR) ─────────────────────────────────────────────────
    dict(species='Apis mellifera',        folder='Apis_mellifera',           functional_group='Pollinators',            medium='AIR',  translocation=False),
    dict(species='Bombus terrestris',     folder='Bombus_terrestris',        functional_group='Pollinators',            medium='AIR',  translocation=False),
    dict(species='Bombus lapidarius',     folder='Bombus_lapidarius',        functional_group='Pollinators',            medium='AIR',  translocation=False),
    dict(species='Xylocopa violacea',     folder='Xylocopa_violacea',        functional_group='Pollinators',            medium='AIR',  translocation=False),

    # ── 6. Lepidoptera (AIR) ─────────────────────────────────────────────────
    dict(species='Papilio machaon',       folder='Papilio_machaon',          functional_group='Lepidoptera',            medium='AIR',  translocation=False),
    dict(species='Aglais urticae',        folder='Aglais_urticae',           functional_group='Lepidoptera',            medium='AIR',  translocation=False),
    dict(species='Vanessa cardui',        folder='Vanessa_cardui',           functional_group='Lepidoptera',            medium='AIR',  translocation=False),
    dict(species='Zerynthia rumina',      folder='Zerynthia_rumina',         functional_group='Lepidoptera',            medium='AIR',  translocation=False),

    # ── 7. Vegetation (LAND matrix) ──────────────────────────────────────────
    dict(species='Quercus ilex',          folder='Quercus_ilex',             functional_group='Vegetation',             medium='LAND', translocation=True),
    dict(species='Cistus ladanifer',      folder='Cistus_ladanifer',         functional_group='Vegetation',             medium='LAND', translocation=False),
    dict(species='Erica arborea',         folder='Erica_arborea',            functional_group='Vegetation',             medium='LAND', translocation=False),
    dict(species='Thymus zygis',          folder='Thymus_zygis',             functional_group='Vegetation',             medium='LAND', translocation=False),
    dict(species='Salvia rosmarinus',     folder='Salvia_rosmarinus',        functional_group='Vegetation',             medium='LAND', translocation=False),
    dict(species='Ficus carica',          folder='Ficus_carica',             functional_group='Vegetation',             medium='LAND', translocation=False),
    dict(species='Lavandula stoechas',    folder='Lavandula_stoechas',       functional_group='Vegetation',             medium='LAND', translocation=False),
    dict(species='Cytisus scoparius',     folder='Cytisus_scoparius',        functional_group='Vegetation',             medium='LAND', translocation=False),
    dict(species='Rosa canina',           folder='Rosa_canina',              functional_group='Vegetation',             medium='LAND', translocation=False),
    # Quercus suber → no GBIF Portugal dataset (listed in transloc_spp.md as no GBIF records)

    # ── 8. Decomposers (LAND) ────────────────────────────────────────────────
    dict(species='Scarabaeus sacer',      folder='Scarabaeus_sacer',         functional_group='Decomposers',            medium='LAND', translocation=False),
    dict(species='Nicrophorus vespillo',  folder='Nicrophorus_vespillo',     functional_group='Decomposers',            medium='LAND', translocation=False),

    # ── 9. Invasive Species (AIR) ────────────────────────────────────────────
    dict(species='Vespa velutina',        folder='Vespa_velutina',           functional_group='Invasive Species',       medium='AIR',  translocation=False),
    # Vespa mandarinia — included in transloc_spp.md but NOT in repo-architecture; skip if absent
]

print(f'Species in registry: {len(SPECIES_REGISTRY)}')

Species in registry: 57


## Load All Species

Preference order for each species folder:
1. `.parquet` (GeoParquet — fastest, preserves geometry)
2. `.gpkg` (GeoPackage — full spatial, EPSG:3035)
3. `.csv` (fallback, no geometry)

Failed loads are logged to `load_errors` and skipped.

In [9]:
gdfs        = []
load_log    = []   # (species, path_used, n_records)
load_errors = []   # (species, reason)

def _try_load(entry: dict) -> gpd.GeoDataFrame | None:
    """Try to load a species GeoDataFrame from the processed directory."""
    folder   = entry['folder']
    sp_dir   = PROCESSED_SPECIES / folder

    # Candidate paths in preference order
    candidates = [
        sp_dir / f'{folder}.parquet',
        sp_dir / f'{folder}.gpkg',
        sp_dir / f'{folder}.csv',
    ]

    for path in candidates:
        if not path.exists():
            continue
        try:
            if path.suffix == '.parquet':
                gdf = gpd.read_parquet(path)
            elif path.suffix == '.gpkg':
                gdf = gpd.read_file(path)
            else:  # .csv — no geometry
                df  = pd.read_csv(path)
                gdf = gpd.GeoDataFrame(df)   # geometry will be None/missing
            return gdf, path
        except Exception as exc:
            load_errors.append((entry['species'], str(path), str(exc)))
            continue

    return None, None


for entry in SPECIES_REGISTRY:
    gdf, path_used = _try_load(entry)

    if gdf is None:
        load_errors.append((entry['species'], 'No readable file found in species directory', ''))
        print(f"  ✗  {entry['species']:45s}  — not found")
        continue

    # Attach metadata columns
    gdf['species_name']     = entry['species']
    gdf['functional_group'] = entry['functional_group']
    gdf['medium']           = entry['medium']
    gdf['translocation']    = entry['translocation']

    gdfs.append(gdf)
    load_log.append((entry['species'], str(path_used), len(gdf)))
    print(f"  ✓  {entry['species']:45s}  {len(gdf):>7,} records  [{path_used.suffix}]")

print(f'\nLoaded {len(gdfs)} / {len(SPECIES_REGISTRY)} species')

  ✓  Canis lupus                                         83 records  [.parquet]
  ✓  Lynx pardinus                                        1 records  [.parquet]
  ✓  Felis silvestris                                     2 records  [.parquet]
  ✓  Vulpes vulpes                                    2,544 records  [.parquet]
  ✓  Genetta genetta                                    616 records  [.parquet]
  ✓  Herpestes ichneumon                                658 records  [.parquet]
  ✓  Martes foina                                       849 records  [.parquet]
  ✓  Mustela putorius                                    77 records  [.parquet]
  ✓  Cervus elaphus                                     630 records  [.parquet]
  ✓  Capreolus capreolus                                638 records  [.parquet]
  ✓  Capra pyrenaica                                     12 records  [.parquet]
  ✓  Sus scrofa                                       3,864 records  [.parquet]
  ✓  Lepus europaeus                    

## Load Summary

In [10]:
log_df = pd.DataFrame(load_log, columns=['species', 'path', 'n_records'])
display(log_df)

if load_errors:
    print('\n⚠️  Load errors:')
    err_df = pd.DataFrame(load_errors, columns=['species', 'path', 'error'])
    display(err_df)

,species,path,n_records
0,Canis lupus,/home/linda/Documents/myData/data-management/d...,83
1,Lynx pardinus,/home/linda/Documents/myData/data-management/d...,1
2,Felis silvestris,/home/linda/Documents/myData/data-management/d...,2
3,Vulpes vulpes,/home/linda/Documents/myData/data-management/d...,2544
4,Genetta genetta,/home/linda/Documents/myData/data-management/d...,616
5,Herpestes ichneumon,/home/linda/Documents/myData/data-management/d...,658
6,Martes foina,/home/linda/Documents/myData/data-management/d...,849
7,Mustela putorius,/home/linda/Documents/myData/data-management/d...,77
8,Cervus elaphus,/home/linda/Documents/myData/data-management/d...,630
9,Capreolus capreolus,/home/linda/Documents/myData/data-management/d...,638


## Concatenate into Master GeoDataFrame

In [12]:
master = gpd.GeoDataFrame(
    pd.concat(gdfs, ignore_index=True)
)

# Ensure consistent CRS — reproject anything that differs from EPSG:4326
# (parquet files from the pipeline are stored in EPSG:4326; gpkg in EPSG:3035)
TARGET_CRS = 'EPSG:4326'

if master.crs is None:
    master = master.set_crs(TARGET_CRS)
elif master.crs.to_epsg() != 4326:
    master = master.to_crs(TARGET_CRS)

print('Master GeoDataFrame:')
print(f'  Shape  : {master.shape}')
print(f'  CRS    : {master.crs}')
print(f'  Columns: {list(master.columns)}')
master.head()

Master GeoDataFrame:
  Shape  : (75067, 16)
  CRS    : {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Hor

,gbifID,species,decimalLatitude,decimalLongitude,countryCode,eventDate,basisOfRecord,coordinateUncertaintyInMeters,year,month,day,geometry,species_name,functional_group,medium,translocation
0,6185652144,Canis lupus,39.553570,-8.800753,PT,2021-01-02T12:43:29Z,HUMAN_OBSERVATION,NaN,2021.0,1.0,2.0,POINT (-8.80075 39.55357),Canis lupus,Terrestrial Carnivores,LAND,False
1,6185251549,Canis lupus,41.077606,-8.601757,PT,2026-03-20T17:22:04Z,HUMAN_OBSERVATION,4.0,2026.0,3.0,20.0,POINT (-8.60176 41.07761),Canis lupus,Terrestrial Carnivores,LAND,False
2,6185026082,Canis lupus,41.868138,-7.985825,PT,2026-03-29T17:14:54,HUMAN_OBSERVATION,336.0,2026.0,3.0,29.0,POINT (-7.98582 41.86814),Canis lupus,Terrestrial Carnivores,LAND,False
3,6179384907,Canis lupus,37.346921,-7.468764,PT,2026-03-08T11:41Z,HUMAN_OBSERVATION,31.0,2026.0,3.0,8.0,POINT (-7.46876 37.34692),Canis lupus,Terrestrial Carnivores,LAND,False
4,6159544997,Canis lupus,40.619247,-7.665688,PT,2023-12-06T16:35Z,HUMAN_OBSERVATION,27.0,2023.0,12.0,6.0,POINT (-7.66569 40.61925),Canis lupus,Terrestrial Carnivores,LAND,False


## Record Counts by Functional Group

In [13]:
summary = (
    master
    .groupby(['functional_group', 'medium'])
    .agg(
        n_species  = ('species_name', 'nunique'),
        n_records  = ('species_name', 'count'),
        transloc_n = ('translocation', 'sum')
    )
    .reset_index()
    .sort_values('functional_group')
)

display(summary)

,functional_group,medium,n_species,n_records,transloc_n
0,Aquatic & Freshwater,WATER,13,1591,453
1,Birds,AIR,9,27973,8665
2,Decomposers,LAND,2,28,0
3,Invasive Species,AIR,1,2064,0
4,Large Herbivores,LAND,7,5890,2004
5,Lepidoptera,AIR,4,8857,0
6,Pollinators,AIR,4,7008,0
7,Terrestrial Carnivores,LAND,8,4830,1
8,Vegetation,LAND,9,16826,220


## Split into Functional Group GeoDataFrames

Each group is stored both as a variable and exported to  
`data/processed/groups/<group_slug>.gpkg`.

In [14]:
def _slug(name: str) -> str:
    """Convert group name to a filesystem-safe slug."""
    return name.lower().replace(' ', '_').replace('&', 'and').replace('/', '_')


group_gdfs = {}

for group in master['functional_group'].unique():
    gdf_group = master[master['functional_group'] == group].copy()
    slug      = _slug(group)
    group_gdfs[slug] = gdf_group

    # Export GeoPackage
    out_path = PROCESSED_GROUPS / f'{slug}.gpkg'
    gdf_group.to_file(out_path, driver='GPKG')

    print(f'  {group:35s}  {len(gdf_group):>7,} records  → {out_path.name}')

print(f'\n{len(group_gdfs)} functional group files written to {PROCESSED_GROUPS}')

  Terrestrial Carnivores                 4,830 records  → terrestrial_carnivores.gpkg
  Large Herbivores                       5,890 records  → large_herbivores.gpkg
  Birds                                 27,973 records  → birds.gpkg
  Aquatic & Freshwater                   1,591 records  → aquatic_and_freshwater.gpkg
  Pollinators                            7,008 records  → pollinators.gpkg
  Lepidoptera                            8,857 records  → lepidoptera.gpkg
  Vegetation                            16,826 records  → vegetation.gpkg
  Decomposers                               28 records  → decomposers.gpkg
  Invasive Species                       2,064 records  → invasive_species.gpkg

9 functional group files written to /home/linda/Documents/myData/data-management/data/processed/groups


## Named Variables for Each Group

Convenient references for downstream notebooks.

In [15]:
gdf_carnivores   = group_gdfs.get('terrestrial_carnivores')
gdf_herbivores   = group_gdfs.get('large_herbivores')
gdf_birds        = group_gdfs.get('birds')
gdf_aquatic      = group_gdfs.get('aquatic_and_freshwater')
gdf_pollinators  = group_gdfs.get('pollinators')
gdf_lepidoptera  = group_gdfs.get('lepidoptera')
gdf_vegetation   = group_gdfs.get('vegetation')
gdf_decomposers  = group_gdfs.get('decomposers')
gdf_invasive     = group_gdfs.get('invasive_species')

# Quick sanity check
for name, gdf in [
    ('gdf_carnivores',  gdf_carnivores),
    ('gdf_herbivores',  gdf_herbivores),
    ('gdf_birds',       gdf_birds),
    ('gdf_aquatic',     gdf_aquatic),
    ('gdf_pollinators', gdf_pollinators),
    ('gdf_lepidoptera', gdf_lepidoptera),
    ('gdf_vegetation',  gdf_vegetation),
    ('gdf_decomposers', gdf_decomposers),
    ('gdf_invasive',    gdf_invasive),
]:
    n = len(gdf) if gdf is not None else 0
    print(f'  {name:22s}  {n:>7,} records')

  gdf_carnivores            4,830 records
  gdf_herbivores            5,890 records
  gdf_birds                27,973 records
  gdf_aquatic               1,591 records
  gdf_pollinators           7,008 records
  gdf_lepidoptera           8,857 records
  gdf_vegetation           16,826 records
  gdf_decomposers              28 records
  gdf_invasive              2,064 records


## Translocation Subset

In [16]:
gdf_translocation = master[master['translocation'] == True].copy()

print(f'Translocation species records: {len(gdf_translocation):,}')
print(f'Species: {sorted(gdf_translocation["species_name"].unique())}')

# Export
trans_path = PROCESSED_GROUPS / 'translocation_species.gpkg'
gdf_translocation.to_file(trans_path, driver='GPKG')
print(f'\nExported → {trans_path}')

Translocation species records: 11,343
Species: ['Achondrostoma occidentale', 'Capreolus capreolus', 'Cervus elaphus', 'Iberochondrostoma lusitanicum', 'Lepus granatensis', 'Lynx pardinus', 'Margaritifera margaritifera', 'Oryctolagus cuniculus', 'Pandion haliaetus', 'Porphyrio porphyrio', 'Quercus ilex', 'Squalius alburnoides', 'Squalius aradensis', 'Squalius pyrenaicus']

Exported → /home/linda/Documents/myData/data-management/data/processed/groups/translocation_species.gpkg


## Export Master Parquet

For fast loading in downstream notebooks.

In [17]:
master_parquet = PROJECT_ROOT / 'data' / 'processed' / 'all_species_master.parquet'
master.to_parquet(master_parquet)
print(f'Master GeoParquet written → {master_parquet}')
print(f'Total records: {len(master):,}')

Master GeoParquet written → /home/linda/Documents/myData/data-management/data/processed/all_species_master.parquet
Total records: 75,067


## Done

**Outputs written:**

| File | Description |
|------|-------------|
| `data/processed/all_species_master.parquet` | Full master GeoDataFrame (EPSG:4326) |
| `data/processed/groups/terrestrial_carnivores.gpkg` | Group 1 |
| `data/processed/groups/large_herbivores.gpkg` | Group 2 |
| `data/processed/groups/birds.gpkg` | Group 3 |
| `data/processed/groups/aquatic_and_freshwater.gpkg` | Group 4 |
| `data/processed/groups/pollinators.gpkg` | Group 5 |
| `data/processed/groups/lepidoptera.gpkg` | Group 6 |
| `data/processed/groups/vegetation.gpkg` | Group 7 |
| `data/processed/groups/decomposers.gpkg` | Group 8 |
| `data/processed/groups/invasive_species.gpkg` | Group 9 |
| `data/processed/groups/translocation_species.gpkg` | All translocation species combined |

**Variables available in memory:**
`master`, `gdf_carnivores`, `gdf_herbivores`, `gdf_birds`, `gdf_aquatic`,  
`gdf_pollinators`, `gdf_lepidoptera`, `gdf_vegetation`, `gdf_decomposers`,  
`gdf_invasive`, `gdf_translocation`